In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold
from sklearn import svm
from sklearn.metrics import confusion_matrix

In [ ]:
def standardize(flare_data):
  median = []
  std = []
  flare_data = np.array(flare_data)
  n_elements = flare_data.shape[0]
  
  for j in range(flare_data.shape[1]):
    standard_deviation_of_this_feature = np.std(flare_data[:, j])
    median_of_this_feature = np.median(flare_data[:, j])
    std.append(standard_deviation_of_this_feature)
    median.append(median_of_this_feature)

    for i in range(n_elements):
        flare_data[i, j] = (flare_data[i, j] - median_of_this_feature) / (standard_deviation_of_this_feature)
  return flare_data, median, std

def fit_data(flare_data, median, std):
    flare_data = np.array(flare_data)
    for j in range(flare_data.shape[1]):
        flare_data[:, j] = (flare_data[:, j] - median[j]) / std[j]
    return flare_data

In [ ]:
data_12 = pd.read_csv('SHARP_with_pil_param_12_a.csv')
data_24 = pd.read_csv('SHARP_with_pil_param_24_a.csv')
data_36 = pd.read_csv('SHARP_with_pil_param_36_a.csv')
data_48 = pd.read_csv('SHARP_with_pil_param_48_a.csv')

In [ ]:
data_12

### Using this data to train a model

In [ ]:
data_12.columns

In [ ]:
data_dict = {12:data_12, 24:data_24, 36:data_36, 48:data_48}
for window in [12,24,36,48]:
    data = data_dict[window]
    pil_col = [x for x in data.columns if x.endswith("PIL") or x == "CME"]
    data =data[pil_col]
    data.dropna(inplace=True)
    X = data.drop('CME', axis=1)
    y = data["CME"]

    # Feature Ranking
    N_features = X.shape[1]
    selector = SelectKBest(f_classif, k=N_features)
    selector.fit(X, y)
    scores = selector.scores_

    feature_names = X.columns
    scores_df = pd.DataFrame({'Feature': feature_names, 'Score': scores}).sort_values('Score', ascending=False)
    ordered_features = scores_df['Feature'].tolist()

    plt.figure(figsize=(8, 7), dpi = 300)
    plt.barh(scores_df['Feature'], scores_df['Score'])
    plt.xlabel('Scores')
    plt.ylabel('Features')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

    pos = data[data['CME'] == 1].copy()
    pos = pos[ordered_features]
    neg = data[data['CME'] == 0].copy()
    neg = neg[ordered_features]

    min_a = pos.index.min()
    max_a = pos.index.max()

    if max_a != min_a:
        pos['normalized_event_number'] = (pos.index - min_a) / (max_a - min_a)
    else:
        pos['normalized_event_number'] = 0

    min_b = neg.index.min()
    max_b = neg.index.max()

    if max_b != min_b:
        neg['normalized_event_number'] = (neg.index - min_b) / (max_b - min_b)
    else:
        neg['normalized_event_number'] = 0

    a = 1
    plt.figure(figsize=(7, 5),dpi =300)

    for i in range(1):
        mean_pos = np.mean(pos.iloc[:, i])
        mean_neg = np.mean(neg.iloc[:, i])
        std_pos = np.std(pos.iloc[:, i])
        std_neg = np.std(neg.iloc[:, i])

        plt.hist(pos.iloc[:, i], bins=20, density=True, alpha=0.7)
        plt.hist(neg.iloc[:, i], bins=20, density=True, alpha=0.7)
        plt.axvline(mean_pos, color='b',ls='-', label=f"Mean($\\mu$) Pos: {mean_pos:.2e}")
        plt.axvline(mean_neg, color='r',ls='--', label=f"Mean($\\mu$) Neg: {mean_neg:.2e}")
        ymin, ymax = plt.ylim()
        plt.fill_betweenx([ymin, ymax], mean_pos - std_pos, mean_pos + std_pos, color='b', alpha=0.2,label=f'Stdev($\\sigma$) Pos: {std_pos:.2e}')
        plt.fill_betweenx([ymin, ymax], mean_neg - std_neg, mean_neg + std_neg, color='r', alpha=0.2, label=f'Stdev($\\sigma$) Neg: {std_neg:.2e}')
        plt.xlabel(ordered_features[i])
        plt.ylabel('Probability Density')
        plt.legend()

        a += 2

    plt.tight_layout()
    plt.show()

In [ ]:
data_dict = {12:data_12, 24:data_24, 36:data_36, 48:data_48}
for window in [12,24,36,48]:
    data = data_dict[window]
    data.dropna(inplace=True)
    data = data.drop(['Unnamed: 0', 'HARPNUM', 'NOAA', 'Class', 'T_REC', 'Peak Time'], axis=1)
    X = data.drop('CME', axis=1)
    y = data["CME"]

    # Feature Ranking
    N_features = X.shape[1]
    selector = SelectKBest(f_classif, k=N_features)
    selector.fit(X, y)
    scores = selector.scores_

    feature_names = X.columns
    scores_df = pd.DataFrame({'Feature': feature_names, 'Score': scores}).sort_values('Score', ascending=False)
    ordered_features = scores_df['Feature'].tolist()

    plt.figure(figsize=(8, 7))
    plt.barh(scores_df['Feature'], scores_df['Score'])
    plt.xlabel('Scores')
    plt.ylabel('Features')
    plt.title('F Score')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

    # Spread visualisation
    pos = data[data['CME'] == 1].copy()
    pos = pos[ordered_features]
    neg = data[data['CME'] == 0].copy()
    neg = neg[ordered_features]

    min_a = pos.index.min()
    max_a = pos.index.max()

    if max_a != min_a:
        pos['normalized_event_number'] = (pos.index - min_a) / (max_a - min_a)
    else:
        pos['normalized_event_number'] = 0

    min_b = neg.index.min()
    max_b = neg.index.max()

    if max_b != min_b:
        neg['normalized_event_number'] = (neg.index - min_b) / (max_b - min_b)
    else:
        neg['normalized_event_number'] = 0

    a = 1
    plt.figure(figsize=(16, 90),dpi = 300)

    for i in range(len(feature_names) - 1):
        mean_pos = np.mean(pos.iloc[:, i])
        mean_neg = np.mean(neg.iloc[:, i])
        std_pos = np.std(pos.iloc[:, i])
        std_neg = np.std(neg.iloc[:, i])

        plt.subplot(len(feature_names), 2, a)
        plt.scatter(pos["normalized_event_number"], pos.iloc[:, i], alpha=0.7, label="Positive (Eruptive Flares)")
        plt.scatter(neg["normalized_event_number"], neg.iloc[:, i], alpha=0.7, label="Negative (Confined Flares)")
        plt.axhline(mean_pos, color='b', ls='-', label=f"Mean($\\mu$) Pos: {mean_pos:.2f}")
        plt.axhline(mean_neg, color='r', ls='--', label=f"Mean($\\mu$) Neg: {mean_neg:.2f}")
        plt.xlabel("Normalized Event Number")
        plt.ylabel(ordered_features[i])
        plt.legend()

        plt.subplot(len(feature_names), 2, a + 1)
        plt.hist(pos.iloc[:, i], bins=20, density=True, alpha=0.7)
        plt.hist(neg.iloc[:, i], bins=20, density=True, alpha=0.7)
        plt.axvline(mean_pos, color='b',ls='-', label=f"Mean($\\mu$) Pos: {mean_pos:.2f}")
        plt.axvline(mean_neg, color='r',ls='--', label=f"Mean($\\mu$) Neg: {mean_neg:.2f}")
        ymin, ymax = plt.ylim()
        plt.fill_betweenx([ymin, ymax], mean_pos - std_pos, mean_pos + std_pos, color='b', alpha=0.2,label=f'Stdev($\\sigma$) Pos: {std_pos:.2f}')
        plt.fill_betweenx([ymin, ymax], mean_neg - std_neg, mean_neg + std_neg, color='r', alpha=0.2, label=f'Stdev($\\sigma$) Neg: {std_neg:.2f}')
        plt.xlabel(ordered_features[i])
        plt.ylabel('Probability Density')
        plt.legend()

        a += 2

    plt.tight_layout()
    plt.show()

In [ ]:
data_dict = {12:data_12, 24:data_24, 36:data_36, 48:data_48}
for window in [12,24,36,48]:
    data = data_dict[window]
    data.dropna(inplace=True)
    print(data.isna().sum().sum())
    data = data.drop(['Unnamed: 0', 'HARPNUM', 'NOAA', 'Class', 'T_REC', 'Peak Time'], axis=1)
        
    PIL_cols = [col for col in data.columns if col.endswith("_PIL")]
    AR_cols = [col for col in data.columns if col not in PIL_cols and col !='CME']
    PIL_AR_cols= [col for col in data.columns if col != 'CME']

    param_sets = {'PILs':PIL_cols, 'ARs':AR_cols, 'PILARs':PIL_AR_cols}
    for set_name, cols in param_sets.items():
        tss_mean = []
        tss_std = []
        param_C = []
        param_gamma = []
        skf = StratifiedKFold(n_splits=10, shuffle=True,random_state=42)
        for C in [1e-3, 1e-2, 1e-1, 1, 1e1, 1e2, 1e3]:
            for gamma in [1e-4, 1e-3, 1e-2, 1e-1, 1, 10]:
                tss_scores = []

                X_ml = np.array(data[cols])
                y_ml = np.array(data['CME'])
                for train_index, test_index in skf.split(X_ml, y_ml):
                    xtrain, xtest = X_ml[train_index], X_ml[test_index]
                    ytrain, ytest = y_ml[train_index], y_ml[test_index]

                    xtrain_norm, median, std = standardize(xtrain)
                    xtest_norm = fit_data(xtest, median, std)

                    clf = svm.SVC(class_weight="balanced", C=C, gamma=gamma, kernel='rbf', probability=True)
                    clf.fit(xtrain_norm, ytrain)
                    ypred = clf.predict(xtest_norm)

                    TN, FP, FN, TP = confusion_matrix(ytest, ypred).ravel()

                    if (TP + FN == 0 or FP + TN == 0):
                        tss_scores.append(np.nan)
                        continue

                    # TSS
                    tss = TP / (TP + FN) - FP / (FP + TN)
                    tss_scores.append(tss)

                # Store average scores
                tss_mean.append(round(np.nanmean(tss_scores), 4))
                tss_std.append(round(np.nanstd(tss_scores), 4))
                param_C.append(C)
                param_gamma.append(gamma)

                print(f"{set_name} completed with C={C} and gamma={gamma}")
        results_df = pd.DataFrame({'C':param_C,'gamma':param_gamma,'TSS_mean':tss_mean,'TSS_std':tss_std})
        csv_path = f"result/result_a_{window}_ml_results_{window}_{set_name}.csv"
        print(f"Saved: {csv_path}")

In [ ]:
result = pd.read_csv('result/result_a_12_ml_results_12_ARs.csv')
result.head(1)

In [ ]:
# Loading the results
timestep = np.array([12,24,36,48])
for j,offset in zip(["ARs","PILARs","PILs"],[-1.2,0,1.2]):
    mean_TSS = []
    std_TSS = []
    for i in timestep:
        result = pd.read_csv(f"result/result_a_{i}_ml_results_{i}_{j}.csv")
        best_tss_row = result.loc[result['TSS_mean'].idxmax()]
        print(best_tss_row)
        mean_TSS.append(best_tss_row["TSS_mean"])
        std_TSS.append(best_tss_row["TSS_std"])
    plt.errorbar(timestep + offset, mean_TSS, yerr=std_TSS,linestyle='None',marker = "o", capsize=0,label=j)
    plt.title(j)
    plt.xticks(timestep)
    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.xlabel("Time (hours)")
    plt.ylabel("TSS")
    plt.legend()


In [ ]:
data_dict = {12:data_12, 24:data_24, 36:data_36, 48:data_48}

fig_data = {'ARs': {}, 'PILs': {}, 'PILARs': {}}

for window in [12, 24, 36, 48]:
    data = data_dict[window].copy()
    data.dropna(inplace=True)
    print(data.isna().sum().sum())
    data = data.drop(['Unnamed: 0', 'HARPNUM', 'NOAA', 'Class', 'T_REC', 'Peak Time'], axis=1)

    PIL_cols    = [col for col in data.columns if col.endswith("_PIL")]
    AR_cols     = [col for col in data.columns if col not in PIL_cols and col != 'CME']
    PIL_AR_cols = [col for col in data.columns if col != 'CME']
    param_sets  = {'PILs': PIL_cols, 'ARs': AR_cols, 'PILARs': PIL_AR_cols}

    for set_name, cols in param_sets.items():
        selector = SelectKBest(f_classif, k=len(cols))
        selector.fit(data[cols].to_numpy(), data['CME'].to_numpy())
        scores_df        = pd.DataFrame({'Feature': cols, 'Score': selector.scores_}) \
                             .sort_values('Score', ascending=False)
        ordered_features = scores_df['Feature'].tolist()
        num_features     = list(range(1, len(ordered_features) + 1))

        skf      = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
        tss_mean = []
        tss_std  = []

        for k in num_features:
            selected_features = ordered_features[:k]
            X_ml = data[selected_features].to_numpy()
            y_ml = np.array(data['CME'])

            best_tss_k, best_std_k = -np.inf, 0                           

            for C in [1e-3, 1e-2, 1e-1, 1, 1e1, 1e2, 1e3]:               
                for gamma in [1e-4, 1e-3, 1e-2, 1e-1, 1, 10]:             
                    tss_scores = []
                    for train_index, test_index in skf.split(X_ml, y_ml):
                        xtrain, xtest = X_ml[train_index], X_ml[test_index]
                        ytrain, ytest = y_ml[train_index], y_ml[test_index]
                        xtrain_norm, median, std = standardize(xtrain)
                        xtest_norm = fit_data(xtest, median, std)
                        clf = svm.SVC(class_weight="balanced", C=C, gamma=gamma, kernel='rbf', probability=False)
                        clf.fit(xtrain_norm, ytrain)
                        ypred = clf.predict(xtest_norm)
                        TN, FP, FN, TP = confusion_matrix(ytest, ypred).ravel()
                        if (TP + FN == 0 or FP + TN == 0):
                            tss_scores.append(np.nan)
                            continue
                        tss_scores.append(TP / (TP + FN) - FP / (FP + TN))

                    val = round(np.nanmean(tss_scores), 4)
                    s   = round(np.nanstd(tss_scores),  4)
                    if val > best_tss_k:                                   
                        best_tss_k, best_std_k = val, s                    

            tss_mean.append(best_tss_k)                                    
            tss_std.append(best_std_k)                                     

        fig_data[set_name][window] = (num_features, tss_mean, tss_std)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(10, 7), dpi=300)
gs = gridspec.GridSpec(2, 2, figure=fig)

axes = [
    fig.add_subplot(gs[0, 0]),   # Top left
    fig.add_subplot(gs[0, 1]),   # Top right
    fig.add_subplot(gs[1, :])    # Bottom spanning both columns
]

set_names = ['ARs', 'PILs', 'PILARs']

for ax, set_name in zip(axes, set_names):
    for window in [12, 24, 36, 48]:
        num_features, tss_mean, tss_std = fig_data[set_name][window]
        x = np.array(num_features, dtype=float)

        ax.errorbar(
            x, tss_mean, yerr=tss_std,
            marker='o', linestyle='-',
            capsize=0,
            label=f'{window} h'
        )

    ax.set_title(set_name)
    ax.set_xticks(range(1, len(num_features)+1))
    ax.set_yticks(np.arange(0, 0.6, 0.1))
    ax.set_xlabel("Number of Parameters")
    ax.set_ylabel("TSS")

# One legend for the whole figure
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc='lower center',
    ncol=4,
    bbox_to_anchor=(0.5, -0.02),
    frameon=False
)

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()